# Import libraries

In [ ]:
import pandas as pd
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import threading
from typing import Tuple
from tqdm import tqdm

import sys
sys.path.append('..')
from utils.audio_util import convert_mp3_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy

# Moving files to new directory

In [3]:
df = pd.read_csv("../data/raw/thai-central/thai-central_mapping.csv")

In [4]:
AUDIO_BASE_DIR = "../data/raw/thai-central/audio_v2"
DEST_DIR = "../data/converted/thai-central-to-vctk"
AUDIO_DEST_DIR = os.path.join(DEST_DIR, "wav16")
TXT_DEST_DIR = os.path.join(DEST_DIR, "txt")

In [5]:
# Add full path column
df['full_path'] = df['public_name'].apply(lambda x: os.path.join(AUDIO_BASE_DIR, x))

# Filter existing files
df_filtered = df[df['full_path'].apply(os.path.exists)].copy()

# Count files per speaker
speaker_counts = df_filtered['speaker_id'].value_counts()
valid_speakers = speaker_counts[speaker_counts >= 100].index

# Filter speakers with >= 100 files
df_filtered = df_filtered[df_filtered['speaker_id'].isin(valid_speakers)]

In [ ]:
# Create new speaker ID mapping
sorted_speakers = speaker_counts[speaker_counts >= 100].sort_values().index
speaker_mapping = {
    spk: f'tc{i+1:04d}' 
    for i, spk in enumerate(sorted_speakers)
}

# Add new speaker ID column
df_filtered['new_speaker_id'] = df_filtered['speaker_id'].map(speaker_mapping)

In [13]:
df_train = pd.read_csv("../data/raw/thai-central/train.csv")
df_dev = pd.read_csv("../data/raw/thai-central/dev.csv")

df_all = pd.concat([df_train, df_dev], ignore_index=True)
df_all['sentence'] = df_all['sentence'].apply(lambda x: "".join(x.split()))
df_all['audio'] = df_all['audio'].apply(lambda x: os.path.join(AUDIO_BASE_DIR, x))
df_all

,utterance,sentence,audio
0,thai-central_000000,ทีมจากอิสราเอลไม่ควรได้เป็นเจ้าบ้านในเกมยูฟ่าคัพ,../data/raw/thai-central/audio_v2/train_audio1...
1,thai-central_000001,แต่พอไหมอะไรคือแต้อีบ็อบฮ่าฮ่ากูพิมพ์ผิดไหมล่ะ...,../data/raw/thai-central/audio_v2/train_audio0...
2,thai-central_000003,ทุกสิ่งทุกอย่างจะราบรื่น,../data/raw/thai-central/audio_v2/train_audio0...
3,thai-central_000005,เร็วหันมองเวลาตั้งกระทู้,../data/raw/thai-central/audio_v2/train_audio1...
4,thai-central_000006,มีขนาดหนาและใหญ่กว่าเกร็ดปลาทั่วไปจนเหมือนเครื...,../data/raw/thai-central/audio_v2/train_audio0...
...,...,...,...
341134,thai-central_433292,มีของทั้งหมดเป็นจำนวนหนึ่งหมื่นหนึ่งพันกระป๋องค่ะ,../data/raw/thai-central/audio_v2/dev_audio00/...
341135,thai-central_433304,บ้านงิ้วงามหมู่สี่มีอาณาเขตติดต่อกับหมู่บ้านใก...,../data/raw/thai-central/audio_v2/dev_audio00/...
341136,thai-central_433457,กองทัพเรือหมายถึงกองกำลังทางทหารที่ปฏิบัติการท...,../data/raw/thai-central/audio_v2/dev_audio00/...
341137,thai-central_433700,กรมอู่ทหารเรือ,../data/raw/thai-central/audio_v2/dev_audio00/...


In [14]:
audio2sentence = dict(zip(df_all['audio'], df_all['sentence']))

In [ ]:
# Thread-safe set for character collection
all_chars = set()
chars_lock = threading.Lock()

# Thread-safe list for tracking skipped files
skip_files = []
skip_lock = threading.Lock()

def process_file_pair(args: Tuple[str, str, str, str]) -> None:
    """Process a single pair of audio and text files"""
    speaker_id, src_path, dest_audio_path, dest_txt_path = args
    try:
        # Create speaker directories
        speaker_wav_dir = os.path.join(AUDIO_DEST_DIR, speaker_id)
        speaker_txt_dir = os.path.join(TXT_DEST_DIR, speaker_id)
        os.makedirs(speaker_wav_dir, exist_ok=True)
        os.makedirs(speaker_txt_dir, exist_ok=True)
        
        # Process audio
        dest_filename = os.path.splitext(os.path.basename(dest_audio_path))[0] + '_mic1.flac'
        dest_path = os.path.join(speaker_wav_dir, dest_filename)
        
        if not convert_mp3_to_flac(src_path, dest_path):
            raise Exception("Failed to convert audio")
        
        # Create empty text file and collect characters
        base_filename = os.path.splitext(dest_filename)[0]
        txt_filename = f"{base_filename}.txt"
        txt_path = os.path.join(speaker_txt_dir, txt_filename)
        
        # In this case we're creating empty text files
        # Modify this part if you need to process actual text content
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(audio2sentence[src_path])

        # Collect characters
        with chars_lock:
            all_chars.update(audio2sentence[src_path])
            
    except Exception as e:
        print(f"Error processing file {src_path}: {e}")
        with skip_lock:
            skip_files.append(src_path)

# Remove existing directories if they exist
if os.path.exists(DEST_DIR):
    print("Clearing destination folder")
    shutil.rmtree(DEST_DIR)

# Create necessary directories
os.makedirs(AUDIO_DEST_DIR, exist_ok=True)
os.makedirs(TXT_DEST_DIR, exist_ok=True)

# Create processing arguments
process_args = [
    (row['new_speaker_id'], row['full_path'], 
        os.path.join(AUDIO_DEST_DIR, row['new_speaker_id'], os.path.basename(row['public_name'])),
        os.path.join(TXT_DEST_DIR, row['new_speaker_id'], os.path.basename(row['public_name'])))
    for _, row in df_filtered.iterrows()
]

# Process files in parallel with progress bar
max_workers = os.cpu_count()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    list(tqdm(
        executor.map(process_file_pair, process_args),
        total=len(process_args),
        desc=f"Processing files (using {max_workers} workers)"
    ))

# Print results
print(f"Processed {len(df_filtered) - len(skip_files)} file pairs")
print(f"Skipped {len(skip_files)} pairs")
print(f"Unique characters found: {''.join(sorted(all_chars))}")

Clearing destination folder


Processing files (using 16 workers): 100%|██████████| 225028/225028 [50:25<00:00, 74.37it/s] 

Processed 225028 file pairs
Skipped 0 pairs
Unique characters found: กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรฤลวศษสหฬอฮฯะัาำิีึืุูเแโใไ็่้๊๋์


# Resample, trim, and normalize audio

In [16]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/thai-central-to-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav16 to wav16_silence_trimmed
src_dir = "../data/converted/thai-central-to-vctk/wav16"
dst_dir = "../data/converted/thai-central-to-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [17]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 8

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 225028 files...


100%|██████████| 225028/225028 [03:29<00:00, 1073.76it/s]

Done !


In [ ]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

In [ ]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(dst_dir, "flac", -27, 16000)

In [ ]:
DEST_DIR = Path(DEST_DIR)

# Write character files
sorted_chars = sorted(all_chars)
with open(DEST_DIR / 'all_chars_unicode.txt', 'w') as f:
   f.write(''.join(c.encode('unicode_escape').decode('ascii') for c in sorted_chars))
   
with open(DEST_DIR / 'all_chars.txt', 'w') as f:
   f.write(''.join(sorted_chars))